# 00 · Data exploration

Simulated strong-lensing images from the DeepLense common test: 150×150 single-channel maps,
min–max normalised to [0, 1], three classes of dark-matter substructure.

| folder | class | meaning |
|---|---|---|
| `no` | 0 | smooth lens, no substructure |
| `sphere` | 1 | spherical CDM-like subhalos |
| `vort` | 2 | vortex substructure (e.g. superfluid / axion DM) |

**Split policy used by every model**
- `train/` (10k per class) → 90 % training, 10 % validation (model selection)
- `val/` (2.5k per class) → **held-out test set**, touched once at the end

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch

from deeplense import CLASS_NAMES
from deeplense.utils import get_device, seed_everything

seed_everything(42)
DEVICE = get_device()
DATA_ROOT = ROOT / "data" / "lensing"
RESULTS = ROOT / "results"
print("device:", DEVICE)

In [ ]:
from deeplense.data import compute_stats, list_split

train_p, train_y = list_split(DATA_ROOT, "train")
test_p, test_y = list_split(DATA_ROOT, "val")
print("train:", np.bincount(train_y), " test:", np.bincount(test_y))

x = np.load(train_p[0])
print("shape", x.shape, "dtype", x.dtype, "range", (x.min(), x.max()))
print("pixel mean / std (3k-image sample):", compute_stats(train_p))

## Examples per class

In [ ]:
rng = np.random.default_rng(0)
train_y = np.asarray(train_y)
fig, axes = plt.subplots(3, 5, figsize=(14, 8.5))
for c, name in enumerate(CLASS_NAMES):
    for j, i in enumerate(rng.choice(np.flatnonzero(train_y == c), 5, replace=False)):
        axes[c, j].imshow(np.load(train_p[i])[0], cmap="inferno")
        axes[c, j].axis("off")
    axes[c, 0].set_title(name, loc="left", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

## Class-mean images and differences

The classes are visually almost identical: substructure is a small perturbation on top of the
main Einstein ring. The mean differences show where, on average, the signal lives.

In [ ]:
means = []
for c in range(3):
    idx = rng.choice(np.flatnonzero(train_y == c), 1000, replace=False)
    means.append(np.mean([np.load(train_p[i])[0] for i in idx], axis=0))

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for c in range(3):
    axes[c].imshow(means[c], cmap="inferno"); axes[c].set_title(f"mean: {CLASS_NAMES[c]}")
for ax, (a, b) in zip(axes[3:], [(1, 0), (2, 0)]):
    d = means[a] - means[b]; v = np.abs(d).max()
    ax.imshow(d, cmap="RdBu_r", vmin=-v, vmax=v); ax.set_title(f"{CLASS_NAMES[a].split()[0]} − {CLASS_NAMES[b].split()[0]}")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## Leakage check

Hash every array and look for exact duplicates within or across splits. A duplicate between
`train/` and `val/` would inflate test metrics. (Takes ~1–2 min: reads all 37.5k files.)

In [ ]:
from deeplense.data import find_duplicates
find_duplicates(DATA_ROOT)

Result on this dataset: **0 duplicate groups** among 37,500 files, so the test split is clean.

**Augmentation.** Lensing images have no preferred orientation, so the 8 symmetries of the
square (90° rotations + flips) are exact invariances. `RandomDihedral` uses only these, avoiding
the interpolation blur of arbitrary-angle rotation, which would smear pixel-scale substructure.